In [ ]:
# Binary Judgment matrix

import logging
import sys
from pathlib import Path

import pandas as pd
from tqdm import tqdm

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s", stream=sys.stdout)
logger = logging.getLogger()

base_output_dir = Path("output")
judgment_matrix_dir = base_output_dir / "Judgment_Matrix"
judgment_matrix_dir.mkdir(parents=True, exist_ok=True)

leak_pressure_dir = base_output_dir / "leak" / "Leak_Pressure"
baseline_pressure_dir = base_output_dir / "baseline" / "Baseline_Pressure"

threshold = 1.0  # 0.6, 1.0, 1.5 m

batch_pairs = []
for leak_file in sorted(leak_pressure_dir.glob("Leak_Pressure_batch*.parquet")):
    batch_id = leak_file.stem.split("batch")[1]
    baseline_file = baseline_pressure_dir / f"Baseline_Pressure_batch{batch_id}.parquet"
    if baseline_file.exists():
        batch_pairs.append((leak_file, baseline_file, batch_id))

if not batch_pairs:
    raise FileNotFoundError(
        f"No matching leak/baseline batch pairs found under {leak_pressure_dir} and {baseline_pressure_dir}"
    )

for leak_file, baseline_file, batch_id in tqdm(batch_pairs, desc="Processing batches"):
    output_file = judgment_matrix_dir / f"Judgment_Matrix_batch{batch_id}.parquet"
    if output_file.exists():
        continue

    try:
        sample = pd.read_parquet(baseline_file, engine="pyarrow").head(1)
        node_columns = [col for col in sample.columns if col != "Iteration"]

        leak_df_iter = pd.read_parquet(leak_file, columns=["Iteration"], engine="pyarrow")
        baseline_df_iter = pd.read_parquet(
            baseline_file, columns=["Iteration"], engine="pyarrow"
        )

        common_iters = sorted(
            set(leak_df_iter["Iteration"].unique()).intersection(
                set(baseline_df_iter["Iteration"].unique())
            )
        )

        chunk_size = 500
        all_judgments = []

        for i in range(0, len(common_iters), chunk_size):
            chunk_iters = common_iters[i : i + chunk_size]
            if not chunk_iters:
                continue

            leak_data = pd.read_parquet(
                leak_file,
                engine="pyarrow",
                columns=["Iteration"] + node_columns,
                filters=[("Iteration", "in", chunk_iters)],
            )
            baseline_data = pd.read_parquet(
                baseline_file,
                engine="pyarrow",
                filters=[("Iteration", "in", chunk_iters)],
            )

            leak_data.sort_values("Iteration", inplace=True)
            baseline_data.sort_values("Iteration", inplace=True)

            judgment = (
                (leak_data[node_columns] - baseline_data[node_columns]).abs() > threshold
            ).astype(int)
            judgment.insert(0, "Iteration", leak_data["Iteration"].astype("int64"))

            all_judgments.append(judgment)
            del leak_data, baseline_data, judgment

        if all_judgments:
            combined_judgment = pd.concat(all_judgments, ignore_index=True)
            combined_judgment["Iteration"] = combined_judgment["Iteration"].astype("int64")
            combined_judgment.sort_values("Iteration", inplace=True)

            combined_judgment.to_parquet(
                output_file,
                engine="pyarrow",
                compression="snappy",
                index=False,
            )

            del combined_judgment, all_judgments

    except Exception as e:
        logger.error(f"Error processing batch {batch_id}: {e}")
        import traceback
        logger.error(traceback.format_exc())

merged_file = judgment_matrix_dir / "Binary_Judgment_Matrix.parquet"
judgment_batch_files = sorted(judgment_matrix_dir.glob("Judgment_Matrix_batch*.parquet"))

if not judgment_batch_files:
    raise FileNotFoundError(f"No judgment matrix batch files found in {judgment_matrix_dir}")

try:
    if len(judgment_batch_files) <= 10:
        all_dfs = []
        for batch_file in tqdm(judgment_batch_files, desc="Reading batch files"):
            df = pd.read_parquet(batch_file, engine="pyarrow")
            df["Iteration"] = df["Iteration"].astype("int64")
            all_dfs.append(df)

        if not all_dfs:
            raise RuntimeError("No dataframes to merge")

        combined = pd.concat(all_dfs, ignore_index=True)
        combined.to_parquet(
            merged_file,
            engine="pyarrow",
            compression="snappy",
            index=False,
        )
        del combined, all_dfs

    else:
        import shutil

        first_file = judgment_batch_files[0]
        shutil.copy(first_file, merged_file)

        chunk_size = 5
        remaining_files = judgment_batch_files[1:]

        with tqdm(total=len(remaining_files), desc="Merging batch files") as pbar:
            for i in range(0, len(remaining_files), chunk_size):
                chunk_files = remaining_files[i : i + chunk_size]

                merged_df = pd.read_parquet(merged_file, engine="pyarrow")
                merged_df["Iteration"] = merged_df["Iteration"].astype("int64")

                for batch_file in chunk_files:
                    batch_df = pd.read_parquet(batch_file, engine="pyarrow")
                    batch_df["Iteration"] = batch_df["Iteration"].astype("int64")
                    merged_df = pd.concat([merged_df, batch_df], ignore_index=True)
                    del batch_df
                    pbar.update(1)

                merged_df.to_parquet(
                    merged_file,
                    engine="pyarrow",
                    compression="snappy",
                    index=False,
                )
                del merged_df

except Exception as e:
    logger.error(f"Error during merge process: {e}")
    import traceback
    logger.error(traceback.format_exc())

In [ ]:
# Node thresholds template

import logging
import sys
from pathlib import Path

import pandas as pd
import wntr

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s", stream=sys.stdout)
logger = logging.getLogger()

inp_file_path = Path("networks/Hanoi.inp")  # Hanoi.inp, C-Town.inp
output_dir = Path("output/config")
output_csv_name = "node_specific_thresholds.csv"


def extract_all_node_ids_to_csv_template(inp_path: Path, csv_path: Path) -> None:
    wn = wntr.network.WaterNetworkModel(str(inp_path))

    node_ids: list[str] = []
    node_ids.extend([str(node_id) for node_id in wn.junction_name_list])
    node_ids.extend([str(node_id) for node_id in wn.reservoir_name_list])
    node_ids.extend([str(node_id) for node_id in wn.tank_name_list])

    unique_node_ids = sorted(set(node_ids))
    if not unique_node_ids:
        raise RuntimeError(f"No node IDs found in model: {inp_path}")

    df_nodes = pd.DataFrame(unique_node_ids, columns=["NodeID"])
    df_nodes["Threshold"] = ""

    csv_path.parent.mkdir(parents=True, exist_ok=True)
    df_nodes.to_csv(csv_path, index=False, quoting=1)
    logger.info(f"Wrote: {csv_path}")


if __name__ == "__main__":
    output_csv_path = output_dir / output_csv_name
    extract_all_node_ids_to_csv_template(inp_file_path, output_csv_path)

In [ ]:
# Mixed threshold binary judgment matrix

import logging
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s", stream=sys.stdout)
logger = logging.getLogger()

output_base_dir = Path("output")
judgment_matrix_dir_mixed = output_base_dir / "Judgment_Matrix_Mixed_Threshold_Corrected"
judgment_matrix_dir_mixed.mkdir(parents=True, exist_ok=True)

leak_pressure_dir = output_base_dir / "leak" / "Leak_Pressure"
baseline_pressure_dir = output_base_dir / "baseline" / "Baseline_Pressure"

node_thresholds_csv_file = output_base_dir / "config" / "node_specific_thresholds.csv"
default_threshold_if_unmapped = 1.0


def load_node_threshold_map_from_csv(csv_path: Path):
    threshold_map = {}
    if not csv_path.exists():
        return threshold_map

    df_thresholds = pd.read_csv(csv_path, dtype={"NodeID": str})
    if "NodeID" not in df_thresholds.columns or "Threshold" not in df_thresholds.columns:
        return threshold_map

    for _, row in df_thresholds.iterrows():
        node_id = row["NodeID"]
        if pd.isna(node_id) or not str(node_id).strip():
            continue
        node_id = str(node_id).strip()

        threshold_val = row["Threshold"]
        if pd.isna(threshold_val) or (
            isinstance(threshold_val, str) and not threshold_val.strip()
        ):
            continue

        try:
            threshold_map[node_id] = float(threshold_val)
        except ValueError:
            continue

    return threshold_map


if __name__ == "__main__":
    start_time = time.time()

    node_threshold_map = load_node_threshold_map_from_csv(node_thresholds_csv_file)

    batch_pairs = []
    for leak_file in sorted(leak_pressure_dir.glob("Leak_Pressure_batch*.parquet")):
        batch_id = leak_file.stem.split("batch")[1]
        baseline_file = baseline_pressure_dir / f"Baseline_Pressure_batch{batch_id}.parquet"
        if baseline_file.exists():
            batch_pairs.append((leak_file, baseline_file, batch_id))

    if not batch_pairs:
        raise FileNotFoundError(
            f"No batch pairs found under {leak_pressure_dir} and {baseline_pressure_dir}"
        )

    first_baseline_sample_df = pd.read_parquet(batch_pairs[0][1], engine="pyarrow").head(1)
    master_node_columns = [
        str(col) for col in first_baseline_sample_df.columns if col != "Iteration"
    ]
    del first_baseline_sample_df

    if not master_node_columns:
        raise RuntimeError("Could not determine node columns from sample baseline Parquet")

    aligned_thresholds = np.array(
        [
            node_threshold_map.get(col_name, default_threshold_if_unmapped)
            for col_name in master_node_columns
        ]
    )

    for leak_file_path, baseline_file_path, batch_id in tqdm(
        batch_pairs, desc="Processing batches (Mixed Threshold)"
    ):
        output_parquet_file = judgment_matrix_dir_mixed / f"MixedThreshold_Judgment_Matrix_batch{batch_id}.parquet"

        if output_parquet_file.exists():
            continue

        try:
            leak_iter_df = pd.read_parquet(
                leak_file_path, columns=["Iteration"], engine="pyarrow"
            )
            baseline_iter_df = pd.read_parquet(
                baseline_file_path, columns=["Iteration"], engine="pyarrow"
            )
            common_iterations = sorted(
                list(
                    set(leak_iter_df["Iteration"].unique())
                    & set(baseline_iter_df["Iteration"].unique())
                )
            )
            del leak_iter_df, baseline_iter_df

            if not common_iterations:
                continue

            chunk_size = 1000
            batch_judgments_list = []

            cols_to_read = ["Iteration"] + master_node_columns

            for i in range(0, len(common_iterations), chunk_size):
                chunk_iterations = common_iterations[i : i + chunk_size]
                if not chunk_iterations:
                    continue

                leak_chunk_df = pd.read_parquet(
                    leak_file_path,
                    engine="pyarrow",
                    columns=cols_to_read,
                    filters=[("Iteration", "in", chunk_iterations)],
                )
                baseline_chunk_df = pd.read_parquet(
                    baseline_file_path,
                    engine="pyarrow",
                    columns=cols_to_read,
                    filters=[("Iteration", "in", chunk_iterations)],
                )

                leak_chunk_df = leak_chunk_df.set_index("Iteration")
                baseline_chunk_df = baseline_chunk_df.set_index("Iteration")

                idx_intersect = leak_chunk_df.index.intersection(baseline_chunk_df.index)
                if idx_intersect.empty:
                    continue

                leak_aligned = leak_chunk_df.loc[idx_intersect, master_node_columns]
                baseline_aligned = baseline_chunk_df.loc[idx_intersect, master_node_columns]

                abs_diff_values = np.abs(leak_aligned.values - baseline_aligned.values)
                judgment_values = (abs_diff_values > aligned_thresholds).astype(np.int8)

                judgment_df_chunk = pd.DataFrame(
                    judgment_values, columns=master_node_columns, index=idx_intersect
                )
                judgment_df_chunk.reset_index(inplace=True)

                batch_judgments_list.append(judgment_df_chunk)

            if batch_judgments_list:
                final_batch_df = pd.concat(batch_judgments_list, ignore_index=True)
                final_batch_df["Iteration"] = final_batch_df["Iteration"].astype("int64")
                final_batch_df.sort_values("Iteration", inplace=True)

                final_batch_df.to_parquet(
                    output_parquet_file,
                    engine="pyarrow",
                    compression="snappy",
                    index=False,
                )

        except Exception as e:
            logger.error(f"Error processing batch {batch_id}: {e}")
            import traceback
            logger.error(traceback.format_exc())

    final_merged_file_path = judgment_matrix_dir_mixed / "MixedThreshold_Binary_Judgment_Matrix.parquet"
    batch_files_to_merge = sorted(judgment_matrix_dir_mixed.glob("MixedThreshold_Judgment_Matrix_batch*.parquet"))

    if batch_files_to_merge:
        all_merged_dfs = []
        for f_path in tqdm(batch_files_to_merge, desc="Reading batch files for final merge"):
            df_batch = pd.read_parquet(f_path, engine="pyarrow")
            df_batch["Iteration"] = df_batch["Iteration"].astype("int64")
            all_merged_dfs.append(df_batch)

        if all_merged_dfs:
            final_combined_df = pd.concat(all_merged_dfs, ignore_index=True)
            final_combined_df["Iteration"] = final_combined_df["Iteration"].astype("int64")
            final_combined_df.drop_duplicates(subset=["Iteration"], keep="first", inplace=True)
            final_combined_df.sort_values("Iteration", inplace=True)

            final_combined_df.to_parquet(
                final_merged_file_path,
                engine="pyarrow",
                compression="snappy",
                index=False,
            )

In [ ]:
# Judgment matrix post-process (drop columns to remove reservoirs and tanks)

import logging
import sys
from pathlib import Path

import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s", stream=sys.stdout)
logger = logging.getLogger()

output_base_dir = Path("output")
input_parquet_file = output_base_dir / "Judgment_Matrix" / "Binary_Judgment_Matrix.parquet"
output_parquet_file = output_base_dir / "Judgment_Matrix" / "Binary_Judgment_Matrix_Processed.parquet"

columns_to_remove = ['1']  # Hanoi
# columns_to_remove = ['R1', 'T1', 'T2', 'T3', 'T4', 'T5', 'T6', 'T7']  # C-Town

try:
    if not input_parquet_file.exists():
        raise FileNotFoundError(f"File not found: {input_parquet_file}")

    df = pd.read_parquet(input_parquet_file, engine="pyarrow")

    for col in columns_to_remove:
        if col in df.columns:
            df.drop(columns=col, inplace=True)

    df["Iteration"] = df["Iteration"].astype("int64")

    df.to_parquet(
        output_parquet_file,
        engine="pyarrow",
        compression="snappy",
        index=False,
    )

    logger.info(f"Wrote: {output_parquet_file}")

except Exception as e:
    logger.error(f"An error occurred: {e}")
    import traceback

    logger.error(traceback.format_exc())
    raise

In [ ]:
# Distance matrix (processed nodes only)

import logging
import sys
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
import wntr

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s", stream=sys.stdout)
logger = logging.getLogger()

output_base_dir = Path("output")
inp_file = Path("networks/Hanoi.inp")  # Hanoi.inp, C-Town.inp
processed_parquet_file = output_base_dir / "Judgment_Matrix" / "Binary_Judgment_Matrix_Processed.parquet"

output_parquet_dir = output_base_dir / "Distance_Matrix"
output_parquet_dir.mkdir(parents=True, exist_ok=True)
output_parquet_file = output_parquet_dir / "Distance_Matrix_Processed.parquet"

wn = wntr.network.WaterNetworkModel(str(inp_file))

G = wn.to_graph()
uG = G.to_undirected()

distances = dict(nx.all_pairs_dijkstra_path_length(uG))
distance_df = pd.DataFrame(
    distances, index=wn.node_name_list, columns=wn.node_name_list
).fillna(float("inf"))

if not processed_parquet_file.exists():
    raise FileNotFoundError(f"Processed judgment matrix not found at: {processed_parquet_file}")

judgment_df = pd.read_parquet(processed_parquet_file, engine="pyarrow").head(1)
node_names = [col for col in judgment_df.columns if col != "Iteration"]
if not node_names:
    raise RuntimeError(f"No node columns found in {processed_parquet_file}")

distance_df = distance_df.reindex(index=node_names, columns=node_names, fill_value=np.inf)

distance_df.to_parquet(
    output_parquet_file,
    engine="pyarrow",
    compression="snappy",
    index=True,
)

logger.info(f"Wrote: {output_parquet_file}")